### Импорты, seed и среда

In [43]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import random
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import sentence_transformers
from sentence_transformers import SentenceTransformer

import faiss

In [139]:
print("faiss:", faiss.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

faiss: 1.13.2
sentence-transformers: 5.3.0


In [45]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

SEED = 42
set_seed(SEED)

In [47]:
os.makedirs("artifacts", exist_ok=True)

### База знаний и первичный анализ

In [ ]:
with open("data/llm.txt", "r", encoding="utf-8") as f:
    llm_text = f.read()

llm_text[:500]

'A large language model (LLM) is a computational model designed to perform natural language processing tasks, especially language generation, using contextual relationships derived from a large set of training data. LLMs can generate, summarize, translate and parse text in a variety of contexts, and are the technological underpinning of modern chatbots. LLMs can accurately mimic natural language patterns because they are trained on collections of human-written text. For the same reason, biased or'

### Чанкинг документов

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 150) -> List[str]:
    sentences = re.split(r'(?<=[.!?]) +', text)

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if len(sentence) > chunk_size:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
                current_chunk = []
                current_length = 0

            for i in range(0, len(sentence), chunk_size - overlap):
                chunks.append(sentence[i:i + chunk_size])
            continue

        if current_length + len(sentence) + (1 if current_chunk else 0) <= chunk_size:
            current_chunk.append(sentence)
            current_length += len(sentence) + (1 if len(current_chunk) > 1 else 0)
        else:
            chunks.append(" ".join(current_chunk))

            overlap_chunk = []
            overlap_len = 0
            for s in reversed(current_chunk):
                if overlap_len + len(s) + (1 if overlap_chunk else 0) <= overlap:
                    overlap_chunk.insert(0, s)
                    overlap_len += len(s) + (1 if len(overlap_chunk) > 1 else 0)
                else:
                    break

            current_chunk = overlap_chunk + [sentence]
            current_length = sum(len(s) for s in current_chunk) + len(current_chunk) - 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [140]:
all_chunks = chunk_text(llm_text)

print(f"Общее количество чанков: {len(all_chunks)}")

Общее количество чанков: 146


In [121]:
all_chunks[0]

'A large language model (LLM) is a computational model designed to perform natural language processing tasks, especially language generation, using contextual relationships derived from a large set of training data. LLMs can generate, summarize, translate and parse text in a variety of contexts, and are the technological underpinning of modern chatbots. LLMs can accurately mimic natural language patterns because they are trained on collections of human-written text.'

In [122]:
all_chunks[1]

"LLMs can accurately mimic natural language patterns because they are trained on collections of human-written text. For the same reason, biased or inaccurate training data can make a LLM's output less reliable. As of 2024, the largest and most capable LLMs are all based on transformer architectures, which can be more efficient and parallelizable than earlier statistical and recurrent neural network models. Research into other architectures, such as state space models, is ongoing."

In [123]:
all_chunks[2]

"Research into other architectures, such as state space models, is ongoing. Models like GPT, BERT, and their successors used these advances to demonstrate emergent behaviors at scale, such as finding specific data from a large data set and compositional reasoning. Benchmark evaluations for LLMs test a model's ability to perform one or more language tasks. Modern LLMs may face comprehensive, multi-task evaluations measuring reasoning, factual accuracy, alignment, and safety."

### Эмбеддинги и индекс FAISS

In [66]:
from sentence_transformers import SentenceTransformer
import faiss

In [128]:
model = SentenceTransformer('all-MiniLM-L12-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [129]:
embeddings = model.encode(all_chunks,
                          show_progress_bar=True,
                          convert_to_numpy=True,
                          normalize_embeddings=True,
                          )

embeddings = np.array(embeddings).astype('float32')
dim = embeddings.shape[1]

print("Размерность эмбеддингов:", dim)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Размерность эмбеддингов: 384


In [130]:
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("Количество векторов:", index.ntotal)

Количество векторов: 167


In [ ]:
def search_top_k(query: str, k: int = 3):
    query_vector = model.encode([query],
                                convert_to_numpy=True,
                                normalize_embeddings=True,
                                ).astype('float32')

    distances, indices = index.search(query_vector, k)

    print(f"Query: {query}")
    print("-" * 30)
    for i in range(k):
        idx = indices[0][i]
        dist = distances[0][i]
        print(f"Chunk #{idx} (Dist: {dist:.3f}):")
        print(f"{all_chunks[idx]}\n")
    print()

In [137]:
test_queries = [
    "What is a large language model?",
    "What are hallucinations?",
    "How is text converted to numbers?",
]

for q in test_queries:
    search_top_k(q, k=3)

Query: What is a large language model?
------------------------------
Chunk #0 (Dist: 0.654):
A large language model (LLM) is a computational model designed to perform natural language processing tasks, especially language generation, using contextual relationships derived from a large set of training data. LLMs can generate, summarize, translate and parse text in a variety of contexts, and are the technological underpinning of modern chatbots. LLMs can accurately mimic natural language patterns because they are trained on collections of human-written text.

Chunk #26 (Dist: 0.637):
The tendency towards larger models is visible in the list of large language models. For example, the training of GPT-2 (i.e. a 1.5-billion-parameter model) in 2019 cost $50,000, while training of the PaLM (i.e. a 540-billion-parameter model) in 2022 cost $8 million, and Megatron-Turing NLG 530B (in 2021) cost around $11 million. The qualifier "large" in "large language model" is inherently vague, as there i

### Контрольные запросы и оценка retrieval

### Небольшой эксперимент с параметрами retrieval

### Обновление базы знаний и переиндексация

### Mini-RAG

### Краткий анализ ошибок